In [2]:
from selenium import webdriver

driver = webdriver.Chrome()

driver.get("https://afisha.yandex.ru/samara/events?page=40")

input() # как страница прогрузится нажать

source = driver.page_source

driver.quit()

In [9]:
from bs4 import BeautifulSoup as BS
import json
import dateparser

events = []

def location(lon, lat):
    return "latitude=%f;longitude=%f" % (lat, lon)

soup = BS(source, features = "lxml")
for s in soup.find_all( class_= "events-list__item" ):
    j = json.loads(s.select_one(".event-card-react")["data-bem"])
    props = j["event-card-react"]["props"]
    url = "https://afisha.yandex.ru" + props["link"]
    #print(j)
    #r = requests.get( url )
    event = {
        "title": props["title"],
        "source_url": url,
        "description": props["title"],
        "price": props.get("ticketsPrice"),
        "cover_img_url": None,
        "location": None,
        "date": None
    }
    #print(props["type"])
    #print(props["tag"])
    if props.get("image"):
        event["cover_img_url"] = props["image"]["url"]

    #print(props["ageLimit"])
    if props.get("place") and props["place"].get("coordinates"):
        event["location"] = location(*props["place"].get("coordinates").values())
    
    if props.get("additionalInfo"):
        date = dateparser.parse(props["additionalInfo"])
        if date:
            event["date"] = str(date)

    events.append(event)


In [10]:
def wrap(item):
    if isinstance(item, str):
        return "'%s'" % item
    if item is None:
        return "NULL"
    return item

def wrapper(container):
    return "(%s)" % ",\n".join(list(map(wrap, container)))

f = open("populate_events.sql", "w", encoding="utf-8")

kys = events[0].keys()

f.write("INSERT INTO Events ")

f.write("(%s)" % ",".join(kys))

f.write("\nVALUES ")

f.write(",\n".join([wrapper(e.values()) for e in events]))

f.write(";")

f.close()